# Notebook 05 — Review Summarisation
## NLP Business Case: Automated Customer Reviews
### Ironhack Bootcamp Project

---

In this notebook, I generate **blog-style articles** for each product category cluster discovered
in Notebook 04. The goal is to synthesise thousands of reviews into a single, coherent narrative
that captures what customers love, what they complain about, and which products stand out.

**Pipeline**:  Structured Facts (Python) → Prompt Builder → Mistral API → Markdown Article

### Methodology: Extractive-Abstractive

| Phase | Tool | What it does |
|-------|------|-------------|
| **Extractive** (Python) | pandas + NumPy | Extracts hard facts from data: top products, sentiment stats, worst-rated items, representative review excerpts |
| **Abstractive** (LLM) | Mistral Medium (NVIDIA API) | Takes structured facts and writes a polished, persuasive blog article |

This two-step approach prevents the LLM from hallucinating products or numbers. Every claim
in the final article is traceable back to the data — the LLM's job is *styling*, not *discovery*.

### Model

| Component | Model | Why |
|-----------|-------|-----|
| **Summarisation** | `mistralai/mistral-medium-3.5-128b` (NVIDIA API) | 128B parameters, excellent text quality, free via existing NVIDIA credits, reasoning_effort='high' for deep analysis, streaming support |

**Data source**: Test split from N01 (Arrow) + `clusters.csv` + `cluster_profiles.json` from N04.

---

## Section 0 — Setup & Environment

In [ ]:
# ── 0.1  Detect execution environment ──────────────────────────────────
# Check whether we are running inside Google Colab or locally.
# This flag controls whether we mount Google Drive and install packages.
try:
    import google.colab          # noqa: F401  (import only for detection)
    IN_COLAB = True              # Running on Colab
except ImportError:
    IN_COLAB = False             # Running locally

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# ── 0.2  Mount Google Drive (Colab only) ───────────────────────────
# Mount Drive so notebooks can read and write persistent data.
# Without this, all files would be lost when the Colab runtime resets.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✓ Google Drive mounted")
else:
    print("→ Running locally — Drive mount not needed")

In [ ]:
# ── 0.3  Install dependencies (Colab only) ─────────────────────────
# Most packages are pre-installed in Colab (pandas, numpy, datasets).
# requests is used for the NVIDIA API HTTP calls.
if IN_COLAB:
    !pip install requests -q
    print("✓ Packages installed")
else:
    print("→ Running locally — packages should be pre-installed")

In [ ]:
# ── 0.4  Imports ───────────────────────────────────────────
# All imports in one cell per project convention.

import os                          # Environment variables, file paths
import json                        # Load cluster_profiles.json
import random                      # For review sampling with seed
import warnings                    # Suppress noisy warnings
import time                        # Rate limiting between API calls

import numpy as np                 # Numerical operations
import pandas as pd                # DataFrames for cluster analysis
import requests                    # HTTP calls to NVIDIA API

from datasets import load_from_disk  # Load Arrow dataset from N01

warnings.filterwarnings("ignore")    # Keep output clean

print("✓ All imports ready")

In [ ]:
# ── 0.5  Path Constants ─────────────────────────────────────
# All paths derive from BASE_DIR so the notebook works on Colab and locally.
# DATASET_DIR  : where N01 saved the Arrow dataset
# OUTPUT_DIR : where N04 saved clusters.csv and cluster_profiles.json
# PLOTS_DIR  : unused in N05 (no visualisations), kept for consistency
# SUMMARIES_DIR : where N05 writes the generated markdown articles

# Base directory — different for Colab vs local
BASE_DIR = os.path.join(
    "/content/drive/MyDrive", "nlp-project", "business-case-01"
) if IN_COLAB else os.getcwd()

# Derived paths
DATASET_DIR       = os.path.join(BASE_DIR, "data", "dataset")      # Arrow from N01
OUTPUT_DIR       = os.path.join(BASE_DIR, "data")                # Clusters from N04
                PLOTS_DIR        = os.path.join(BASE_DIR, "data", "plots")        # plots (unused in N05)
SUMMARIES_DIR   = os.path.join(BASE_DIR, "data", "summaries")    # N05 output

# Ensure output directories exist
os.makedirs(SUMMARIES_DIR, exist_ok=True)

print(f"BASE_DIR       : {BASE_DIR}")
print(f"DATASET_DIR      : {DATASET_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
print(f"SUMMARIES_DIR  : {SUMMARIES_DIR}")

The NVIDIA API key is read from the environment variable — never hardcoded. This keeps the notebook portable across Colab (Secrets panel) and local execution.


In [ ]:
# ── 0.6  NVIDIA API Key ───────────────────────────────────
# Read the NVIDIA API key from the environment variable.
# This is portable across Colab and local execution.
# On Colab: set via the Secrets panel (key icon in left sidebar)
#           or use os.environ['NVIDIA_API_KEY'] = 'your-key' before this cell.
# Locally:   export NVIDIA_API_KEY='your-key' in your terminal.
#
# Expected output: API key loaded (masked) or clear error message.

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if NVIDIA_API_KEY:
    # Mask the key for display — never print secrets
    masked = NVIDIA_API_KEY[:6] + "..." + NVIDIA_API_KEY[-4:]
    print(f"✓ NVIDIA API key loaded: {masked}")
else:
    print("✗ NVIDIA_API_KEY not found!")
    print("  Set it with: os.environ['NVIDIA_API_KEY'] = 'your-key'")
    print("  Or on Colab: add 'NVIDIA_API_KEY' in the Secrets panel")
    print("  Or locally:   export NVIDIA_API_KEY='your-key'")

I set all global constants here: random seed, extractive sampling parameters, and Mistral API settings. Keeping them in one place makes tuning easier.


In [ ]:
# ── 0.7  Global Constants ───────────────────────────────────
# RANDOM_SEED ensures reproducible review sampling and API responses.
# REVIEWS_PER_CLUSTER controls how many representative reviews to extract.
# API settings control the LLM generation behaviour.

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Extractive phase settings
N_TOP_PRODUCTS       = 5     # How many top products to include per cluster
N_REVIEW_SAMPLES     = 3     # Representative reviews per cluster (1 pos, 1 neg, 1 neu)
MIN_PRODUCT_REVIEWS  = 5     # Minimum reviews for a product to be considered in ranking

# API settings
API_MODEL       = "mistralai/mistral-medium-3.5-128b"
API_ENDPOINT    = "https://integrate.api.nvidia.com/v1/chat/completions"
API_TEMPERATURE = 0.70                  # Balanced: creative enough for blog tone
API_TOP_P       = 1.00                  # Full probability mass (default)
API_MAX_TOKENS  = 16384                 # Generous: up to 32K allowed, 16K is plenty
API_STREAM      = False                 # False = simpler response parsing in notebook
API_TIMEOUT     = 180                   # Extra time for reasoning_effort='high'

print(f"RANDOM_SEED         : {RANDOM_SEED}")
print(f"N_TOP_PRODUCTS      : {N_TOP_PRODUCTS}")
print(f"N_REVIEW_SAMPLES    : {N_REVIEW_SAMPLES}")
print(f"API Model           : {API_MODEL}")
print(f"API Temperature     : {API_TEMPERATURE}")
print(f"API Top-p           : {API_TOP_P}")
print(f"API Max Tokens      : {API_MAX_TOKENS}")
print(f"API Stream          : {API_STREAM}")

---
## Section 1 — Data Loading

I load three pieces of data produced by the earlier notebooks:

1. **Test split** (Arrow, N01) — review texts, ratings, categories, and product IDs
2. **Cluster assignments** (`clusters.csv`, N04) — which cluster each review belongs to
3. **Cluster profiles** (`cluster_profiles.json`, N04) — metadata per cluster: top terms, sentiment distribution, avg rating

I merge them into a single DataFrame so the extractive phase can slice by cluster.

In [ ]:
# ── 1.1  Load test split from N01 Arrow dataset ──────────────────────
# N01 saved a DatasetDict with keys 'train', 'validation', 'test'.
# I only load the test split — it is the right size for generating summaries.
# Expected output: Dataset with ~60-75K reviews (15% of balanced total).

dataset = load_from_disk(DATASET_DIR)   # Reloads the DatasetDict saved by N01
test_dataset = dataset["test"]         # Only the test split
df_test = test_dataset.to_pandas()     # Convert to DataFrame for easier analysis

print(f"Test split loaded: {len(df_test):,} reviews")
print(f"Columns: {list(df_test.columns)}")
print(f"\nFirst 3 rows:")
display(df_test.head(3))

In [ ]:
# ── 1.2  Load cluster assignments and profiles from N04 ────────────────
# clusters.csv maps each review to a cluster ID (0…k-1).
# cluster_profiles.json has structured metadata per cluster.
# Expected output: both files loaded; cluster count printed.

clusters_path   = os.path.join(OUTPUT_DIR, "clusters.csv")
profiles_path   = os.path.join(OUTPUT_DIR, "cluster_profiles.json")

# Load cluster assignments
df_clusters = pd.read_csv(clusters_path)
print(f"Loaded clusters: {df_clusters.shape[0]:,} rows, {df_clusters.shape[1]} columns")
print(f"Columns: {list(df_clusters.columns)}")

# Load cluster profiles (JSON)
with open(profiles_path, "r") as f:
    cluster_profiles = json.load(f)

N_CLUSTERS = cluster_profiles["n_clusters"]
silhouette = cluster_profiles["silhouette_score"]
print(f"\nClusters found  : {N_CLUSTERS}")
print(f"Silhouette Score : {silhouette:.4f}")
print(f"Total reviews    : {cluster_profiles['total_reviews']:,}")

In [ ]:
# ── 1.3  Merge test data with cluster assignments ───────────────────
# I merge on index to attach cluster IDs to each review.
# The resulting DataFrame is the single source of truth for the extractive phase.
# Expected output: merged DataFrame with cluster column.

# Build the index-to-cluster mapping from the clusters DataFrame.
# N04 saved cluster assignments aligned with the test split index.
if "index" in df_clusters.columns:
    df_clusters = df_clusters.set_index("index")

# Merge clusters into test DataFrame
df_merged = df_test.copy()
df_merged["cluster"] = df_clusters["cluster"]

# Verify cluster distribution
cluster_counts = df_merged["cluster"].value_counts().sort_index()
print("Cluster distribution:")
for cid, count in cluster_counts.items():
    pct = 100 * count / len(df_merged)
    print(f"  Cluster {int(cid):>2}: {count:>8,} reviews ({pct:.1f}%)")

print(f"\nTotal merged     : {len(df_merged):,} reviews")
print(f"Columns available : {list(df_merged.columns)}")

---
## Section 2 — Extractive Phase

This is the **data-to-facts** step. For each cluster, I extract structured insights
using plain Python (pandas, NumPy) — no LLM involved here. This keeps the facts grounded
in real data and prevents the LLM from hallucinating products or numbers.

What I extract per cluster:
- **Size & sentiment** — how many reviews, what percentage are positive/neutral/negative
- **Top products** — most reviewed products (by `parent_asin`), with their average rating
- **Worst product** — the product with the lowest average rating (minimum N reviews)
- **Best product** — the product with the highest average rating (minimum N reviews)
- **Original categories** — which Amazon categories ended up in this cluster
- **Representative reviews** — one positive, one negative, one neutral excerpt per cluster

In [ ]:
# ── 2.1  Per-cluster statistics ──────────────────────────────
# I build a structured facts dictionary per cluster.
# This will feed directly into the prompt template in Section 3.
# Expected output: dictionary with cluster_id as keys, structured data as values.

cluster_facts = {}  # {cluster_id: {size, sentiment_dist, avg_rating, categories, ...}}

for cluster_id in range(N_CLUSTERS):
    mask = df_merged["cluster"] == cluster_id
    cluster_df = df_merged[mask]
    n_reviews = len(cluster_df)

    # Sentiment distribution (from original labels: 0=Neg, 1=Neu, 2=Pos)
    sent_counts = cluster_df["label"].value_counts().to_dict()
    sent_pct = {
        "negative": round(100 * sent_counts.get(0, 0) / n_reviews, 1),
        "neutral":  round(100 * sent_counts.get(1, 0) / n_reviews, 1),
        "positive": round(100 * sent_counts.get(2, 0) / n_reviews, 1),
    }

    # Average rating
    avg_rating = round(float(cluster_df["rating"].mean()), 2)

    # Category distribution (what Amazon categories are in this cluster?)
    cat_dist = cluster_df["category"].value_counts().head(5).to_dict()

    cluster_facts[cluster_id] = {
        "cluster_id":   cluster_id,
        "n_reviews":    n_reviews,
        "pct_of_total": round(100 * n_reviews / len(df_merged), 1),
        "avg_rating":   avg_rating,
        "sentiment":    sent_pct,
        "categories":   cat_dist,
    }

    print(f"Cluster {cluster_id}: {n_reviews:,} reviews | "
          f"Avg rating {avg_rating:.2f} | "
          f"Pos {sent_pct['positive']}% / Neu {sent_pct['neutral']}% / Neg {sent_pct['negative']}%")

print(f"\n✓ Cluster statistics extracted for {N_CLUSTERS} clusters")

In [ ]:
# ── 2.2  Product analysis per cluster ──────────────────────────
# For each cluster, I identify:
#   - Top N products by review count (most popular)
#   - Worst product (lowest avg rating, minimum MIN_PRODUCT_REVIEWS reviews)
#   - Best product (highest avg rating, minimum MIN_PRODUCT_REVIEWS reviews)
# I use parent_asin as the product identifier.
# Expected output: product rankings added to cluster_facts.

for cluster_id in range(N_CLUSTERS):
    cluster_df = df_merged[df_merged["cluster"] == cluster_id].copy()

    # Product-level aggregation
    product_stats = cluster_df.groupby("parent_asin").agg(
        n_reviews=("rating", "count"),
        avg_rating=("rating", "mean"),
        pct_negative=("label", lambda x: 100 * (x == 0).mean()),
        pct_positive=("label", lambda x: 100 * (x == 2).mean()),
    ).reset_index()

    # Top N most-reviewed products
    top_products = product_stats.nlargest(N_TOP_PRODUCTS, "n_reviews")
    cluster_facts[cluster_id]["top_products"] = [
        {
            "product_id":  row["parent_asin"],
            "n_reviews":   int(row["n_reviews"]),
            "avg_rating":  round(float(row["avg_rating"]), 2),
            "pct_positive": round(float(row["pct_positive"]), 1),
            "pct_negative": round(float(row["pct_negative"]), 1),
        }
        for _, row in top_products.iterrows()
    ]

    # Products with enough reviews to be meaningful
    qualified = product_stats[product_stats["n_reviews"] >= MIN_PRODUCT_REVIEWS]

    if len(qualified) > 0:
        # Worst product (lowest avg rating)
        worst = qualified.nsmallest(1, "avg_rating").iloc[0]
        cluster_facts[cluster_id]["worst_product"] = {
            "product_id":  worst["parent_asin"],
            "n_reviews":   int(worst["n_reviews"]),
            "avg_rating":  round(float(worst["avg_rating"]), 2),
        }

        # Best product (highest avg rating)
        best = qualified.nlargest(1, "avg_rating").iloc[0]
        cluster_facts[cluster_id]["best_product"] = {
            "product_id":  best["parent_asin"],
            "n_reviews":   int(best["n_reviews"]),
            "avg_rating":  round(float(best["avg_rating"]), 2),
        }
    else:
        cluster_facts[cluster_id]["worst_product"] = None
        cluster_facts[cluster_id]["best_product"] = None

    # Product count summary
    total_products = product_stats["parent_asin"].nunique()
    print(f"Cluster {cluster_id}: {total_products:,} unique products | "
          f"Top product has {cluster_facts[cluster_id]['top_products'][0]['n_reviews']} reviews")

print(f"\n✓ Product analysis complete for {N_CLUSTERS} clusters")

In [ ]:
# ── 2.3  Representative review samples per cluster ───────────────────
# I extract one representative review of each sentiment type per cluster.
# These excerpts help the LLM understand the tone and vocabulary of the cluster.
# Each excerpt is truncated to 300 characters to keep the prompt concise.
# Expected output: samples added to cluster_facts.

MAX_EXCERPT_LEN = 300  # Truncate review text to this many characters

for cluster_id in range(N_CLUSTERS):
    cluster_df = df_merged[df_merged["cluster"] == cluster_id]

    samples = {}
    for sentiment_label, sentiment_name in [(0, "negative"), (1, "neutral"), (2, "positive")]:
        subset = cluster_df[cluster_df["label"] == sentiment_label]
        if len(subset) > 0:
            # Pick a random review with the given sentiment
            row = subset.sample(n=1, random_state=RANDOM_SEED).iloc[0]
            text = str(row["text"])
            excerpt = text[:MAX_EXCERPT_LEN] + ("…" if len(text) > MAX_EXCERPT_LEN else "")
            samples[sentiment_name] = {
                "rating":  int(row["rating"]),
                "excerpt": excerpt,
            }
        else:
            samples[sentiment_name] = None

    cluster_facts[cluster_id]["review_samples"] = samples

    # Print availability summary
    available = [k for k, v in samples.items() if v is not None]
    print(f"Cluster {cluster_id}: sample reviews available for {', '.join(available)}")

print(f"\n✓ Review samples extracted for {N_CLUSTERS} clusters")

In [ ]:
# ── 2.4  Merge top terms from N04 cluster profiles ─────────────────
# N04 already extracted top TF-IDF terms per cluster.
# I merge them into the facts dictionary so the LLM can use them.
# Expected output: top_terms added to cluster_facts.

for profile in cluster_profiles["clusters"]:
    cid = profile["cluster_id"]
    cluster_facts[cid]["top_terms"] = profile["top_terms"]
    print(f"Cluster {cid}: top terms = {', '.join(profile['top_terms'])}")

print(f"\n✓ Top terms merged from N04 profiles")

In [ ]:
# ── 2.5  Verify extracted facts structure ────────────────────────
# Quick sanity check: print the keys available for cluster 0.
# Expected output: list of keys in the cluster_facts dictionary.

print("Facts structure per cluster (example: cluster 0):")
print("-" * 50)
for key, value in cluster_facts[0].items():
    if isinstance(value, dict):
        print(f"  {key}: {{...}} ({len(value)} entries)")
    elif isinstance(value, list):
        print(f"  {key}: [...] ({len(value)} items)")
    else:
        print(f"  {key}: {value}")

print(f"\n✓ {N_CLUSTERS} clusters ready for prompt building")

---
## Section 3 — Prompt Building

This is where structured facts become a **prompt for the LLM**. The prompt template tells
Mistral exactly what kind of article to write, what structure to use, and what tone to adopt.

### Prompt design principles

1. **No system role**: the NVIDIA API for this model only accepts `user` and `assistant` roles.
   The persona and instructions are prepended to the user message instead.
2. **Structured facts** are embedded in the user message as a data block.
3. **Explicit instructions** control the output: markdown format, section structure, word count.
4. **Constraints** prevent hallucination: the LLM must cite numbers from the data, not invent any.

The prompt is built per cluster so each article is unique to its data.

In [ ]:
# ── 3.1  Prompt builder function ─────────────────────────────
# This function takes the structured facts for one cluster and returns
# a complete prompt (system + user message) ready for the API.
#
# Expected output: a dictionary with "messages" (system + user) and estimated token count.

def build_prompt(facts: dict) -> dict:
    """Build a Mistral API prompt from structured cluster facts.

    Args:
        facts: Dictionary with keys: cluster_id, n_reviews, avg_rating,
               sentiment, top_products, worst_product, best_product,
               top_terms, categories, review_samples.

    Returns:
        dict with keys: 'messages' (list of role/content dicts) and
        'cluster_id' (int).
    """

    cid = facts["cluster_id"]

    # ── Persona instructions (prepended to user message — no system role) ───
    persona = (
        "[ROLE] You are a senior market analyst at a top-tier consulting firm. "
        "Your writing is clear, data-driven, and persuasive. "
        "You write blog articles for business executives who need to understand "
        "what customers think about a product category — not just the numbers, "
        "but the story behind them. "
        "Always cite specific numbers from the data provided. "
        "Never make up product names, ratings, or statistics that are not in the data."
    )

    # ── Build the data block (truncated product table for readability) ────
    products_str = ""
    for i, p in enumerate(facts.get("top_products", []), 1):
        products_str += (
            f"  {i}. Product {p['product_id']}: "
            f"{p['n_reviews']} reviews, "
            f"avg rating {p['avg_rating']:.1f}/5, "
            f"{p['pct_positive']}% positive, "
            f"{p['pct_negative']}% negative\n"
        )

    worst_str = ""
    if facts.get("worst_product"):
        wp = facts["worst_product"]
        worst_str = (
            f"  Product {wp['product_id']}: "
            f"{wp['n_reviews']} reviews, avg rating {wp['avg_rating']:.1f}/5\n"
        )
    else:
        worst_str = "  Not enough data to determine.\n"

    best_str = ""
    if facts.get("best_product"):
        bp = facts["best_product"]
        best_str = (
            f"  Product {bp['product_id']}: "
            f"{bp['n_reviews']} reviews, avg rating {bp['avg_rating']:.1f}/5\n"
        )
    else:
        best_str = "  Not enough data to determine.\n"

    # Categories
    categories_str = ", ".join(facts.get("categories", {}).keys())

    # Review excerpts
    excerpts = ""
    for sentiment in ["positive", "negative", "neutral"]:
        sample = facts.get("review_samples", {}).get(sentiment)
        if sample:
            excerpts += (
                f"  [{sentiment.upper()}] Rating {sample['rating']}/5: "
                f"\"{sample['excerpt']}\"\n"
            )

    # ── User message: data + instructions ──────────────────────────────────
    user_message = f"""Write a blog article analysing customer reviews for the following product category cluster.

## Cluster Data

**Cluster ID**: {cid}
**Total reviews analysed**: {facts['n_reviews']:,}
**Average rating**: {facts['avg_rating']:.2f} / 5
**Percentage of total dataset**: {facts['pct_of_total']}%

### Sentiment Distribution
- Positive: {facts['sentiment']['positive']}%
- Neutral:  {facts['sentiment']['neutral']}%
- Negative: {facts['sentiment']['negative']}%

### Top {len(facts.get('top_products', []))} Most Reviewed Products
{products_str}
### Worst Rated Product (min {MIN_PRODUCT_REVIEWS} reviews)
{worst_str}
### Best Rated Product (min {MIN_PRODUCT_REVIEWS} reviews)
{best_str}
### Amazon Categories Contributing to This Cluster
{categories_str}

### Key Terms
{', '.join(facts.get('top_terms', []))}

### Representative Review Excerpts
{excerpts}

## Instructions

Write a 600–900 word blog article in **Markdown format** with the following sections:

1. **Executive Summary** (2–3 sentences) — the bottom line for a busy executive.
2. **What Customers Love** — highlight the best-rated product and positive review themes.
3. **What Customers Complain About** — the worst-rated product and recurring negative themes.
4. **Sentiment at a Glance** — interpret the sentiment distribution in plain language.
5. **Key Takeaways** — 3–4 actionable insights for a business decision-maker.

**Rules**:
- Use specific numbers and percentages from the data provided above.
- Do NOT invent product names, brands, or statistics not in the data.
- Refer to products by their ID (e.g., "Product B0XXXXX").
- Keep the tone professional but engaging — this is for a business blog.
- Wrap the article in markdown with a `#` title and `##` section headers."""

    # Estimate token count (rough: 1 token ≈ 4 characters for English)
    total_chars = len(user_message)  # persona already included in user_message
    est_tokens = total_chars // 4

    return {
        "cluster_id": cid,
        "messages": [
            {"role": "user", "content": user_message},
        ],
        "est_tokens": est_tokens,
    }

print("✓ Prompt builder function defined")

In [ ]:
# ── 3.2  Preview a prompt (cluster 0) ──────────────────────────
# I build one prompt and print the user message to verify the template.
# This is a sanity check before sending API calls.
# Expected output: the full user message for cluster 0.

preview = build_prompt(cluster_facts[0])
print(f"Cluster {preview['cluster_id']}")
print(f"Estimated prompt tokens: {preview['est_tokens']}")
print(f"Messages: {len(preview['messages'])} (user message, persona embedded)")
print("\n" + "=" * 60)
print("USER MESSAGE PREVIEW:")
print("=" * 60)
print(preview["messages"][0]["content"][:500])
print("... [truncated for display]")

Now I build prompts for all clusters at once. Each prompt includes cluster statistics, top products, representative reviews, and key terms — everything the LLM needs to write a well-informed article.


In [ ]:
# ── 3.3  Build prompts for all clusters ────────────────────────
# Build prompts for every cluster and store them in a list.
# Expected output: list of prompt dicts, one per cluster.

all_prompts = []
for cluster_id in range(N_CLUSTERS):
    prompt_data = build_prompt(cluster_facts[cluster_id])
    all_prompts.append(prompt_data)

total_est_tokens = sum(p["est_tokens"] for p in all_prompts)
print(f"✓ Built {len(all_prompts)} prompts")
print(f"  Estimated total tokens: {total_est_tokens:,}")

---
## Section 4 — Mistral API Calls

This is the **facts-to-article** step. I send each structured prompt to the
Mistral Medium model via NVIDIA's API. The model generates a polished markdown article
that synthesises all the extracted facts into a coherent narrative.

### API details
- **Provider**: NVIDIA NIM (API Catalog)
- **Model**: `mistralai/mistral-medium-3.5-128b` — 128B parameters, `reasoning_effort='high'`
- **Authentication**: Bearer token via `Authorization` header
- **Cost**: Free (NVIDIA provides credits for API Catalog usage)
- **Timeout**: 180 seconds per call (extra time for deep reasoning)

### Error handling
- Network errors: retry once after 5 seconds
- API errors (4xx/5xx): print error details, continue to next cluster
- 202 (async): poll until 200 or timeout

This is the core API wrapper — it sends a prompt to Mistral Medium 3.5 via the NVIDIA API and returns the generated article. I use `reasoning_effort='high'` for deeper analysis at the cost of a few extra seconds per cluster.


In [ ]:
# ── 4.1  API call function ─────────────────────────────────
# This function encapsulates the HTTP call to the NVIDIA API.
# It handles authentication, retries, and error parsing.
# Expected output: generated text (str) or None if the call fails.

def call_mistral_api(prompt_data: dict, max_retries: int = 2) -> dict:
    """Send a prompt to the Mistral model via NVIDIA API.

    Args:
        prompt_data: Dict with 'messages' key (list of role/content dicts).
        max_retries: Number of retries on transient errors.

    Returns:
        dict with keys: 'success' (bool), 'content' (str or None),
        'cluster_id' (int), 'tokens_used' (dict or None), 'error' (str or None).
    """
    cid = prompt_data["cluster_id"]
    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}",
        "Content-Type":  "application/json",
        "Accept":        "application/json",
    }

    payload = {
        "model":            API_MODEL,
        "messages":         prompt_data["messages"],
        "temperature":      API_TEMPERATURE,
        "top_p":            API_TOP_P,
        "max_tokens":       API_MAX_TOKENS,
        "reasoning_effort": "high",
        "seed":             RANDOM_SEED,
        "stream":           API_STREAM,
    }

    for attempt in range(1, max_retries + 2):
        try:
            response = requests.post(
                API_ENDPOINT,
                headers=headers,
                json=payload,
                timeout=API_TIMEOUT,
            )

            if response.status_code == 200:
                data = response.json()
                content = data["choices"][0]["message"]["content"]
                usage   = data.get("usage", {})
                return {
                    "success":     True,
                    "content":     content,
                    "cluster_id":  cid,
                    "tokens_used": usage,
                    "error":       None,
                }

            elif response.status_code == 202:
                # Asynchronous: NVIDIA accepted the request, results pending.
                # In practice, the /chat/completions endpoint usually responds
                # synchronously. This is a fallback.
                print(f"  [Cluster {cid}] Status 202 — async, retrying in 10s (attempt {attempt})")
                time.sleep(10)
                continue

            elif response.status_code == 429:
                # Rate limited — wait and retry
                wait = min(2 ** attempt, 30)  # Exponential backoff capped at 30s
                print(f"  [Cluster {cid}] Rate limited (429) — waiting {wait}s (attempt {attempt})")
                time.sleep(wait)
                continue

            else:
                # Other HTTP errors
                error_msg = f"HTTP {response.status_code}: {response.text[:200]}"
                return {
                    "success":     False,
                    "content":     None,
                    "cluster_id":  cid,
                    "tokens_used": None,
                    "error":       error_msg,
                }

        except requests.exceptions.Timeout:
            print(f"  [Cluster {cid}] Timeout — retrying (attempt {attempt})")
            time.sleep(5)
            continue

        except requests.exceptions.RequestException as e:
            if attempt <= max_retries:
                print(f"  [Cluster {cid}] Network error: {e} — retrying (attempt {attempt})")
                time.sleep(5)
                continue
            else:
                return {
                    "success":     False,
                    "content":     None,
                    "cluster_id":  cid,
                    "tokens_used": None,
                    "error":       str(e),
                }

    # Exhausted retries
    return {
        "success":     False,
        "content":     None,
        "cluster_id":  cid,
        "tokens_used": None,
        "error":       "Exhausted all retry attempts",
    }

print("✓ API call function defined")

In [ ]:
# ── 4.2  Generate articles for all clusters ───────────────────────
# Loop over all clusters and call the Mistral API for each one.
# I add a short delay between calls to avoid rate limiting.
# Expected output: list of result dicts, one per cluster.

if not NVIDIA_API_KEY:
    print("✗ Cannot call API: NVIDIA_API_KEY not set.")
    print("  Please set the environment variable and re-run.")
    api_results = []
else:
    api_results = []
    for i, prompt_data in enumerate(all_prompts):
        cid = prompt_data["cluster_id"]
        print(f"\n[{i+1}/{len(all_prompts)}] Generating article for Cluster {cid} …")
        print(f"  Prompt tokens (est): {prompt_data['est_tokens']:,}")

        result = call_mistral_api(prompt_data)
        api_results.append(result)

        if result["success"]:
            tokens = result.get("tokens_used", {})
            prompt_tok = tokens.get("prompt_tokens", "?")
            completion_tok = tokens.get("completion_tokens", "?")
            total_tok = tokens.get("total_tokens", "?")
            print(f"  ✓ Generated {len(result['content']):,} chars | "
                  f"Tokens: {prompt_tok} prompt + {completion_tok} completion = {total_tok}")
        else:
            print(f"  ✗ FAILED: {result['error']}")

        # Brief delay to respect rate limits
        if i < len(all_prompts) - 1:
            time.sleep(2)

    # Summary
    succeeded = sum(1 for r in api_results if r["success"])
    failed    = len(api_results) - succeeded
    print(f"\n{'=' * 40}")
    print(f"API sweep complete: {succeeded} succeeded, {failed} failed")

---
## Section 5 — Save & Verify Output

I save each generated article as a markdown file and verify the outputs.
The files are saved to `data/summaries/category_*_summary.md` so they are
persistent across Colab sessions (inside Google Drive).

In [ ]:
# ── 5.1  Save generated articles as Markdown files ────────────────────
# Each article is saved to data/summaries/category_{cluster_id}_summary.md.
# Expected output: one .md file per successful API call.

output_files = []

for result in api_results:
    cid = result["cluster_id"]

    if not result["success"]:
        print(f"  Cluster {cid}: SKIPPED (API call failed — {result['error']})")
        continue

    # Build the filename with top terms for readability
    terms = cluster_facts[cid]["top_terms"]
    terms_slug = "_".join(terms[:3])  # First 3 terms for the filename
    filename = f"category_{cid:02d}_{terms_slug}_summary.md"
    filepath = os.path.join(SUMMARIES_DIR, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(result["content"])

    output_files.append(filepath)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  ✓ Cluster {cid:>2} → {filename} ({size_kb:.1f} KB)")

print(f"\n✓ Saved {len(output_files)} articles to {SUMMARIES_DIR}/")

In [ ]:
# ── 5.2  Output file checklist ────────────────────────────────
# Verify that all expected output files were created.

print("Output file checklist:")
print("-" * 60)
all_ok = True

for result in api_results:
    cid = result["cluster_id"]
    if not result["success"]:
        status = f"✗  API FAILED ({result['error'][:40]})"
        all_ok = False
    else:
        terms_slug = "_".join(cluster_facts[cid]["top_terms"][:3])
        filename = f"category_{cid:02d}_{terms_slug}_summary.md"
        filepath = os.path.join(SUMMARIES_DIR, filename)
        exists = os.path.exists(filepath)
        status = "✓  OK" if exists else "✗  MISSING"
        if not exists:
            all_ok = False
    print(f"  Cluster {cid:>2}  {status}")

print("-" * 60)
if all_ok:
    print("All articles generated and saved successfully!")
else:
    print("WARNING: Some articles are missing. Check the API errors above.")

In [ ]:
# ── Final Summary Print ───────────────────────────────────────
# Clean end-of-notebook summary so anyone opening the notebook
# can immediately see what was produced.

succeeded = sum(1 for r in api_results if r["success"])
failed = len(api_results) - succeeded

print("=" * 60)
print("  NOTEBOOK 05 — COMPLETE")
print("  Review Summarisation")
print("=" * 60)
print()
print(f"  Methodology     : Extractive-Abstractive")
print(f"  LLM Model       : {API_MODEL}")
print(f"  API Provider    : NVIDIA NIM")
print(f"  Clusters        : {N_CLUSTERS}")
print(f"  Articles generated: {succeeded} succeeded, {failed} failed")
if succeeded > 0:
    total_tokens = sum(
        r.get("tokens_used", {}).get("total_tokens", 0)
        for r in api_results if r["success"]
    )
    print(f"  Total tokens used: {total_tokens:,}")

for result in api_results:
    cid = result["cluster_id"]
    if result["success"]:
        terms_slug = "_".join(cluster_facts[cid]["top_terms"][:3])
        filename = f"category_{cid:02d}_{terms_slug}_summary.md"
        print(f"  ✓ {filename}")
    else:
        print(f"  ✗ Cluster {cid}: FAILED")

print()
print(f"  Output directory: {SUMMARIES_DIR}/")
print()
print("  ➡️  Next: MASTER_PLAN.md — final report + PDF + PPT")
print("=" * 60)